# 강화학습 26.9.14

In [3]:
mastered_bicycle = False

def try_pedaling():
    import random
    return random.choice(['왼발', '오른발'])

def stayed_balanced():
    import random
    return random.random() > 0.3

def update_strategy(action, reward):
    pass

while not mastered_bicycle:
    action = try_pedaling()
    balanced = stayed_balanced()

    if balanced:
        reward = +10
        print('good job')
    else:
        reward = -5
        print('try again')
    update_strategy(action, reward)

    import random
    if random.random() > 0.95:
        mastered_bicycle = True   # True로! (끝남)

print('success')

good job
good job
good job
good job
success


# 정책

In [ ]:
# class ReinforcementLearningDemo():
#     def __init__(self):
#         self.score = 0
#         self.game_over = False
#     def cartpole_example(self):
#         print('목표 : 막대 쓰러뜨리지 않기')
#         while not self.game_over:
#             state = {
#                 'pole_angle' : 0.1,
#                 'cart_position' : 0.0,
#                 'pole_velocity' : 0.02,
#                 'cart_velocity' : 0.1
#             }
#             print(state)
#             action = self.choose_action(state)
#             print(f'선택한 행동 {'왼쪽' if action == -1 else '오른쪽'}')
#             reward = self.calculate_reward(state, action)
#             print(f'받은 보상 {reward}')
#             self.update_policy(state, action, reward)
#             print('전략 업데이트')
#             self.scroe += reward
#             if self.score < -100:
#                 self.game_over = True
#         print(f'최종 스코어 {self.score}')

#     def choose_action(self, state):
#         import random

#         if random.random() < 0.9:  #단골집에 가는 비율이라 생각하면됨.
#             return 1 if state['pole_angle'] > 0 else -1
#         else:
#             return random.choice([-1, 1])

#     def calculate_reward(self, state, action):
#         if abs(state['pole_angle']) > 0.5:
#             return -100
#         else:
#             return +1

#     def update_policy(self, state, action, reward):
#         learning_rate = 0.1
#         discount_factor = 0.95
#         print('더 나은 전략으로 업데이트')

# demo = ReinforcementLearningDemo()
# demo.cartpole_example()

# QNET

In [5]:
import gym # 강화학습 환경을 만드는것
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

class QNetWork(nn.Module):
    def __init__(self, state_size, action_size, seed, fc1_units = 64, fc2_units = 64):
        super(QNetWork, self).__init__() 
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, fc2_units)
        self.fc3 = nn.Linear(fc2_units, action_size)

    def forward(self,state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
seed = 42
qnetwork_local = QNetWork(state_size, action_size, seed)
qnetwork_target = QNetWork(state_size, action_size, seed)
optimizer = optim.Adam(qnetwork_local.parameters(), lr = 5e-4)

buffer_size = int(1e5)
batch_size = 64
memory = deque(maxlen = buffer_size)

def step(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def sample():
    experiences = random.sample(memory, k=batch_size)
    states = torch.from_numpy(np.vstack([e[0] for e in experiences])).float()      # [...]
    actions = torch.from_numpy(np.vstack([e[1] for e in experiences])).long()      # [...]
    rewards = torch.from_numpy(np.vstack([e[2] for e in experiences])).float()     # [...]
    next_states = torch.from_numpy(np.vstack([e[3] for e in experiences])).float() # [...]
    dones = torch.from_numpy(np.vstack([e[4] for e in experiences]).astype(np.uint8)).float()
    return (states, actions, rewards, next_states, dones)


def learn(experiences, gamma):
    states, actions, rewards, next_states, dones = experiences
    Q_targets_next = qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
    Q_targets = rewards + (gamma * Q_targets_next * (1- dones))
    Q_expected = qnetwork_local(states).gather(1, actions)
    loss = nn.MSELoss()(Q_expected, Q_targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

gamma = 0.99
tau = 1e-3
n_episodes = 200
max_t = 1000

for i_episode in range(1, n_episodes + 1):
    state = env.reset()
    total_reward = 0
    for t in range(max_t):
        state_tensor = torch.from_numpy(state).float().unsqueeze(0)
        with torch.no_grad():
            action_values = qnetwork_local(state_tensor)
        action = np.argmax(action_values.cpu().data.numpy())
        next_state, reward, done, _ = env.step(action)
        step(state, action, reward, next_state, done)
        total_reward += reward
        if len(memory) > batch_size:
            experiences = sample()
            loss = learn(experiences, gamma)
        state = next_state
        if done:
            break

    for target_param, local_param in zip(qnetwork_target.parameters(), qnetwork_local.parameters()):
        target_param.data.copy_(tau * local_param.data + (1 - tau) * target_param.data)
    print(i_episode, total_reward)
env.close()
    

c:\Users\wm032\Documents\17_Deeplearning\venv\lib\site-packages\gym\core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\wm032\Documents\17_Deeplearning\venv\lib\site-packages\gym\wrappers\step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


1 10.0
2 10.0
3 9.0
4 9.0
5 9.0
6 10.0
7 9.0
8 9.0
9 9.0
10 10.0
11 10.0
12 9.0
13 10.0
14 10.0
15 10.0
16 9.0
17 9.0
18 9.0
19 10.0
20 9.0
21 10.0
22 10.0
23 9.0
24 10.0
25 9.0
26 10.0
27 8.0
28 10.0
29 10.0
30 9.0
31 9.0
32 8.0
33 9.0
34 10.0
35 10.0
36 8.0
37 9.0
38 10.0
39 10.0
40 11.0
41 9.0
42 9.0
43 9.0
44 9.0
45 9.0
46 9.0
47 9.0
48 11.0
49 9.0
50 10.0
51 11.0
52 10.0
53 9.0
54 8.0
55 9.0
56 9.0
57 11.0
58 8.0
59 9.0
60 9.0
61 9.0
62 9.0
63 10.0
64 10.0
65 9.0
66 8.0
67 10.0
68 9.0
69 10.0
70 8.0
71 9.0
72 10.0
73 10.0
74 9.0
75 8.0
76 10.0
77 8.0
78 11.0
79 9.0
80 8.0
81 11.0
82 9.0
83 10.0
84 10.0
85 10.0
86 10.0
87 10.0
88 9.0
89 10.0
90 9.0
91 9.0
92 10.0
93 10.0
94 10.0
95 8.0
96 10.0
97 9.0
98 9.0
99 10.0
100 10.0
101 9.0
102 10.0
103 8.0
104 10.0
105 9.0
106 10.0
107 11.0
108 9.0
109 10.0
110 8.0
111 9.0
112 10.0
113 9.0
114 11.0
115 9.0
116 10.0
117 10.0
118 9.0
119 10.0
120 9.0
121 9.0
122 9.0
123 10.0
124 9.0
125 9.0
126 9.0
127 8.0
128 10.0
129 10.0
130 9.0
131 10.0


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

class PolicyNetWork(nn.Module):
    def __init__(self, state_size, action_size, hidden_dim = 128):
        super(PolicyNetWork, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = self.fc2(x)
        return torch.softmax(x, dim = -1)

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

policy_net = PolicyNetWork(state_size, action_size)
optimizer = optim.Adam(policy_net.parameters(), lr = 0.01)

def select_action(state):
    state = torch.from_numpy(state).float().unsqueeze(0)
    probs = policy_net(state)
    action = torch.multinomial(probs, num_samples = 1)
    return action.item(), torch.log(probs[0, action.item()])

def reinforce_update(episode_rewards, episode_log_probs, gamma = 0.99):
    R = 0
    returns = []
    for r in episode_rewards[::-1]:
        R = r + gamma * R
        returns.insert(0, R)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    loss = 0
    for log_prob, R in zip(episode_log_probs, returns):
        loss -= log_prob * R
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

num_episodes = 1000
for episode in range(num_episodes):
    state = env.reset()
    episode_rewards = []
    episode_log_probs = []
    done = False
    while not done:
        action, log_prob = select_action(state)
        next_state, reward, done, _ = env.step(action)
        episode_rewards.append(reward)
        episode_log_probs.append(log_prob)
        state = next_state
    loss = reinforce_update(episode_rewards, episode_log_probs)
    if episode % 50 == 0:
        total_reward = sum(episode_rewards)
        print(episode, total_reward, loss)

env.close()
    


c:\Users\wm032\Documents\17_Deeplearning\venv\lib\site-packages\gym\core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\wm032\Documents\17_Deeplearning\venv\lib\site-packages\gym\wrappers\step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\wm032\Documents\17_Deeplearning\venv\lib\site-packages\gym\utils\passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


0 18.0 -0.1201256513595581
50 142.0 -2.747107744216919
100 195.0 0.1275562047958374
150 194.0 2.714972496032715
200 17.0 -1.8543140888214111
250 500.0 -3.1724932193756104
300 500.0 9.054983139038086
350 265.0 4.825924873352051
400 463.0 -3.4234957695007324
450 500.0 -5.075829029083252
500 280.0 -11.572041511535645
550 500.0 11.25247573852539
600 500.0 8.376579284667969
650 500.0 2.9764342308044434
700 9.0 -0.05268603190779686
750 9.0 0.00012209849955979735
800 10.0 3.361996641615406e-05
850 10.0 2.6485051421332173e-05
900 10.0 5.265350409899838e-05
950 8.0 0.000442460470367223
